## Rasteryzing Building heights and mosaiquing rasters
Rasteryzing buildings at 1m , aligned to the extention of the LST raster.

Mosaiquing trees

### Libraries and general paths

In [ ]:
import rasterio
from rasterio.mask import mask
from rasterio.features import rasterize
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import os
import glob
from rasterio.merge import merge
from shapely.geometry import box, mapping
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.io import MemoryFile

In [ ]:
# user-defined paths
base_dir = '../../../datos-notebooks/1.global-data-cba/'
landsat_image = 'LC08_L2SP_229082_20240125_20240130_02_T1'
buildings_file_name = 'GBA_cba.geojson'
trees_folder_name = 'gch_cba'

In [ ]:
# derived paths

# LST
lst_cropped_path = os.path.join(base_dir, f'LST/LST_cropped_celsius{landsat_image}.tif') # values out of bound are 0
lst_clipped_path = os.path.join(base_dir, f'LST/LST_clipped_celsius{landsat_image}.tif') #values out of bound are nan

#inputs
buildings = os.path.join(base_dir, 'objects-height', buildings_file_name) # vectors of buildings with height column
trees = os.path.join(base_dir, 'objects-height', trees_folder_name) # folder with vector files of trees with height column

# Rasters - outputs
rasters_base = os.path.join(base_dir, 'objects-height/rasterized-1m/')
os.makedirs(rasters_base, exist_ok=True)
building_raster_path = os.path.join(rasters_base, 'buildings_height_1m.tif')
trees_raster_path = os.path.join(rasters_base, 'trees_height_1m.tif')

### Rasterizing biuldings

In [ ]:
# Load LST to get extent and CRS
with rasterio.open(lst_cropped_path) as src_lst:
    lst_crs = src_lst.crs
    bounds = src_lst.bounds  # xmin, ymin, xmax, ymax

# Load buildings
gdf_buildings = gpd.read_file(buildings)
gdf_buildings = gdf_buildings.to_crs(lst_crs)  # match LST CRS

# Define raster grid at 0.5 m
res = 0.5  # pixel size
width = int((bounds.right - bounds.left) / res)
height = int((bounds.top - bounds.bottom) / res)
transform = rasterio.transform.from_origin(bounds.left, bounds.top, res, res)

# Rasterize buildings using 'height' column
building_raster = rasterize(
    [(geom, value) for geom, value in zip(gdf_buildings.geometry, gdf_buildings.height)],
    out_shape=(height, width),
    transform=transform,
    fill=np.nan,
    all_touched=True,
    dtype='float32'
)

# Save raster
with rasterio.open(
    building_raster_path,
    'w',
    driver='GTiff',
    height=building_raster.shape[0],
    width=building_raster.shape[1],
    count=1,
    dtype=building_raster.dtype,
    crs=lst_crs,
    transform=transform
) as dst:
    dst.write(building_raster, 1)

In [ ]:
# Print a window for checking the result
with rasterio.open(building_raster_path) as src:
    window = rasterio.windows.Window(col_off=15000, row_off=7000, width=8000, height=8000)
    data = src.read(1, window=window)

plt.figure(figsize=(8,8))
plt.imshow(data, cmap='viridis')
plt.colorbar(label='Height (m)')
plt.show()

### Mosaiquing trees

reprojecting rasters

In [ ]:
# Read LST CRS and extent (master)
with rasterio.open(lst_cropped_path) as lst:
    lst_crs = lst.crs
    bounds = lst.bounds
    
# Read and reproject tree rasters
tif_files = glob.glob(os.path.join(trees, "*.tif"))
srcs = []

for fp in tif_files:
    src = rasterio.open(fp)
    # Reproject to LST CRS
    transform, width, height = calculate_default_transform(
        src.crs, lst_crs, src.width, src.height, *src.bounds
    )

    memfile = MemoryFile()
    dst = memfile.open(
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=src.dtypes[0],
        crs=lst_crs,
        transform=transform,
    )

    reproject(
        source=rasterio.band(src, 1),
        destination=rasterio.band(dst, 1),
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=transform,
        dst_crs=lst_crs,
        resampling=Resampling.nearest
    )

    srcs.append(dst)
    src.close()


mosaiquing

In [ ]:
with rasterio.open(lst_cropped_path) as src_lst:
    lst_height = src_lst.height
    lst_width = src_lst.width
    lst_transform = src_lst.transform
    lst_crs = src_lst.crs

# Mosaic
mosaic, transform = merge(srcs)

# Clip to LST extent
geom = [mapping(box(bounds.left, bounds.bottom, bounds.right, bounds.top))]

with MemoryFile() as memfile:
    with memfile.open(
        driver="GTiff",
        height=mosaic.shape[1],
        width=mosaic.shape[2],
        count=1,
        dtype=mosaic.dtype,
        crs=lst_crs,
        transform=transform,
    ) as tmp:
        tmp.write(mosaic)
        clipped, clipped_transform = mask(tmp, geom, crop=True)

# aligning to LST grid
aligned = np.zeros((lst_height, lst_width), dtype=clipped.dtype)

reproject(
    source=clipped[0],
    destination=aligned,
    src_transform=clipped_transform,
    src_crs=lst_crs,
    dst_transform=lst_transform,
    dst_crs=lst_crs,
    resampling=Resampling.nearest,
    src_nodata=0,
    dst_nodata=0
)

# Save final raster
out_profile = {
    "driver": "GTiff",
    "height": lst_height,
    "width": lst_width,
    "count": 1,
    "dtype": aligned.dtype,
    "crs": lst_crs,
    "transform": lst_transform,
    "compress": "lzw",
    "nodata": 0,
}

with rasterio.open(trees_raster_path, "w", **out_profile) as dst:
    dst.write(aligned, 1)

# Close sources
for src in srcs:
    src.close()


In [ ]:
with rasterio.open(trees_raster_path) as src:
    print(src.width, src.height, src.res)  # para ver el tamaño real
    data = src.read(1, masked=True)

plt.figure(figsize=(10,10))
plt.imshow(data, cmap='Greens')
plt.colorbar(label='Height (m)')
plt.show()